# 1. One-Week AEDP 148-Household Reconstruction Validation

This notebook validates the AEDP 148-household reconstruction method on a one-week window only. It does not build the full 3-year per-site checkpoints for final aggregation datasets.

## 1.1 Purpose, Inputs, And Outputs

Inputs are the monthly raw AEDP CSV file(s), the recovered 148-site cohort spreadsheet, and existing `ds10`/`ds11` for validation. Outputs are one-week 30-minute per-site checkpoints and audit files under a separate one-week validation workspace.

In [1]:
from pathlib import Path
import re
import sys


def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")


PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")

import pandas as pd
import numpy as np

RAW_AEDP_DIR = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\1. raw\ACAP EDP")
PROCESSED_DIR = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\2. processed")
CLEANED_WORKSPACE = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\3. cleaned\AEDP_different_aggregation")
SITE_LIST_PATH = PROCESSED_DIR / "aedp_cluster_2_2_3years.xlsx"
DS10_PATH = REPO_ROOT / "data" / "ds10_aedp_cluster2_30min.csv"
DS11_PATH = REPO_ROOT / "data" / "ds11_aedp_cluster2_30min_with_weather.csv"

VALIDATION_START = pd.Timestamp("2021-07-01 00:00:00")
VALIDATION_END = pd.Timestamp("2021-07-07 23:30:00")

WINDOW_LABEL = f"{VALIDATION_START:%Y%m%d}_{VALIDATION_END:%Y%m%d}"
ONE_WEEK_WORKSPACE = CLEANED_WORKSPACE / f"one_week_validation_{WINDOW_LABEL}"
SITE_30MIN_DIR = ONE_WEEK_WORKSPACE / "site_30min"
AUDIT_DIR = ONE_WEEK_WORKSPACE / "audit"

for directory in [SITE_30MIN_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RAW_INTERVAL = pd.Timedelta(minutes=5)
RAW_INTERVAL_HOURS = RAW_INTERVAL / pd.Timedelta(hours=1)
INDEX_30MIN = pd.date_range(VALIDATION_START, VALIDATION_END, freq="30min", name="datetime")
INDEX_NATIVE = pd.date_range(VALIDATION_START, VALIDATION_END + pd.Timedelta(minutes=25), freq=RAW_INTERVAL, name="datetime")
CHUNKSIZE = 2_000_000
ROUNDING_TOLERANCE = 1e-3

for required_path in [RAW_AEDP_DIR, SITE_LIST_PATH, DS10_PATH, DS11_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print(f"Validation window: {VALIDATION_START} to {VALIDATION_END}")
print(f"One-week workspace: {ONE_WEEK_WORKSPACE}")
print(f"30-minute rows expected: {len(INDEX_30MIN)}")


Publication project: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1
Repository root: C:\Users\z5404477\Documents\PyNNLF


Validation window: 2021-07-01 00:00:00 to 2021-07-07 23:30:00
One-week workspace: C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\3. cleaned\AEDP_different_aggregation\one_week_validation_20210701_20210707
30-minute rows expected: 336


## 1.2 Recover The 148-Site Cohort

The cohort is read from the recovered cluster spreadsheet. This step confirms the same 148 unique site IDs are available before scanning raw data.

In [2]:
site_list = pd.read_excel(SITE_LIST_PATH)
required = {"edp_site_id", "postcode", "state", "lat_deg", "long_deg", "date_of_first_data", "date_of_last_data", "number_of_days"}
missing = required - set(site_list.columns)
if missing:
    raise ValueError(f"AEDP site list is missing required columns: {sorted(missing)}")
site_list["edp_site_id"] = site_list["edp_site_id"].astype(str)
site_ids = site_list["edp_site_id"].tolist()
if len(site_ids) != 148 or len(set(site_ids)) != 148:
    raise ValueError(f"Expected 148 unique AEDP sites, found {len(site_ids)} rows and {len(set(site_ids))} unique IDs")
site_list.to_csv(AUDIT_DIR / "aedp_148hh_recovered_site_list.csv", index=False)
print(f"Recovered sites: {len(site_ids)}")
display(site_list.head())

Recovered sites: 148


,edp_site_id,postcode,state,lat,long,cluster,lat_deg,long_deg,date_of_first_data,date_of_last_data,number_of_days
0,S0405,2282,NSW,-0.575479,2.646839,2,-32.9725,151.6527,2021-06-28,2024-07-12,1110
1,W0013,2226,NSW,-0.593401,2.636562,2,-33.9994,151.0639,2020-10-13,2024-07-12,1368
2,S0321,2322,NSW,-0.572348,2.647261,2,-32.7931,151.6769,2021-06-03,2024-07-13,1136
3,S0431,2284,NSW,-0.575620,2.646240,2,-32.9806,151.6184,2021-05-27,2024-07-16,1146
4,S0170,2307,NSW,-0.573843,2.647507,2,-32.8788,151.6910,2021-03-08,2024-07-16,1226


## 1.3 Select Monthly Raw Files For The One-Week Window

Raw filenames encode year and month. For example, `edp_data_2021_1254143330.csv` is the December 2021 raw file because the suffix starts with `12`. This notebook scans only files whose encoded month overlaps the validation week.

In [3]:
def encoded_year_month(path: Path) -> pd.Period | None:
    """Return the month encoded in an AEDP monthly raw filename, or None for helper CSVs."""
    match = re.match(r"edp_data_(\d{4})_(\d{2})", path.stem)
    if not match:
        return None
    year = int(match.group(1))
    month = int(match.group(2))
    if not 1 <= month <= 12:
        raise ValueError(f"Invalid encoded month in filename {path.name}: {month}")
    return pd.Period(year=year, month=month, freq="M")


def raw_file_priority(path: Path) -> tuple[int, int, str]:
    """Prefer the organised raw-data folder and avoid Archive copies when duplicate months exist."""
    parts = set(path.parts)
    if "2021 Jul to 2024 Jun" in parts:
        folder_priority = 0
    elif "Archive" in parts:
        folder_priority = 2
    else:
        folder_priority = 1
    return (folder_priority, -len(path.parts), str(path))


def naive_utc_unix(timestamp: pd.Timestamp) -> int:
    return int(timestamp.tz_localize("UTC").timestamp())


raw_files = sorted(RAW_AEDP_DIR.rglob("*.csv"))
if not raw_files:
    raise FileNotFoundError(f"No raw AEDP CSV files found under {RAW_AEDP_DIR}")

raw_file_records = []
for path in raw_files:
    encoded_month = encoded_year_month(path)
    raw_file_records.append({
        "path": path,
        "raw_file": str(path),
        "filename": path.name,
        "relative_path": str(path.relative_to(RAW_AEDP_DIR)),
        "encoded_month": encoded_month,
        "encoded_month_label": None if encoded_month is None else str(encoded_month),
        "size_gb": path.stat().st_size / 1e9,
    })

raw_file_audit = pd.DataFrame(raw_file_records).drop(columns=["path"])
ignored_files = raw_file_audit.loc[raw_file_audit["encoded_month"].isna()].copy()
if not ignored_files.empty:
    ignored_files.to_csv(AUDIT_DIR / "aedp_one_week_ignored_non_monthly_csv_files.csv", index=False)

monthly_records = [record for record in raw_file_records if record["encoded_month"] is not None]
needed_months = set(pd.period_range(VALIDATION_START, INDEX_NATIVE[-1], freq="M"))

selected_records = []
duplicate_records = []
for month in sorted(needed_months):
    candidates = [record for record in monthly_records if record["encoded_month"] == month]
    if not candidates:
        continue
    chosen = sorted(candidates, key=lambda record: raw_file_priority(record["path"]))[0]
    selected_records.append(chosen)
    for record in candidates:
        duplicate_records.append({**record, "chosen_for_month": record["path"] == chosen["path"]})

selected_raw_files = [record["path"] for record in selected_records]
if not selected_raw_files:
    raise FileNotFoundError(f"No monthly raw AEDP CSV files matched months {sorted(map(str, needed_months))}")

duplicate_audit = pd.DataFrame(duplicate_records).drop(columns=["path"])
duplicate_audit.to_csv(AUDIT_DIR / "aedp_one_week_monthly_raw_file_candidates.csv", index=False)
file_audit = pd.DataFrame(selected_records).drop(columns=["path"])
file_audit.to_csv(AUDIT_DIR / "aedp_one_week_selected_raw_files.csv", index=False)
print(f"All CSV files found: {len(raw_files)}")
print(f"Monthly raw CSV files found: {len(monthly_records)}")
print(f"Ignored non-monthly/helper CSV files: {len(ignored_files)}")
print(f"Selected one raw file per required month {sorted(map(str, needed_months))}: {len(selected_raw_files)}")
display(file_audit)


All CSV files found: 112
Monthly raw CSV files found: 109
Ignored non-monthly/helper CSV files: 3
Selected one raw file per required month ['2021-07']: 1


,raw_file,filename,relative_path,encoded_month,encoded_month_label,size_gb
0,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,edp_data_2021_0781620172.csv,2021 Jul to 2024 Jun\edp_data_2021_0781620172.csv,2021-07,2021-07,2.981072


## 1.4 Extract Required Rows From Selected Raw Files

Only the four required columns are read. The scan filters to the 148 recovered sites, `ac_load_net`, and the one-week native 5-minute timestamp range. The filtered 5-minute rows are kept in memory for this validation run; the notebook does not persist 5-minute checkpoints.


In [4]:
usecols = ["edp_site_id", "unix_time", "circuit_label", "real_energy"]
site_id_set = set(site_ids)
start_unix = naive_utc_unix(INDEX_NATIVE[0])
end_unix = naive_utc_unix(INDEX_NATIVE[-1])

raw_scan_rows = []
filtered_chunks = []
for raw_file in selected_raw_files:
    chunk_count = 0
    output_rows = 0
    for chunk in pd.read_csv(raw_file, usecols=usecols, chunksize=CHUNKSIZE):
        chunk_count += 1
        chunk["edp_site_id"] = chunk["edp_site_id"].astype(str)
        chunk = chunk.loc[
            chunk["edp_site_id"].isin(site_id_set)
            & chunk["circuit_label"].eq("ac_load_net")
            & chunk["unix_time"].between(start_unix, end_unix)
        ]
        if chunk.empty:
            continue
        grouped = chunk.groupby(["edp_site_id", "unix_time"], observed=True, as_index=False)["real_energy"].sum()
        output_rows += grouped.shape[0]
        filtered_chunks.append(grouped)
    raw_scan_rows.append({"filename": raw_file.name, "chunks": chunk_count, "output_rows": output_rows})
    print(f"[done] {raw_file.name}: chunks={chunk_count}, rows={output_rows}")

raw_scan_summary = pd.DataFrame(raw_scan_rows)
raw_scan_summary.to_csv(AUDIT_DIR / "aedp_one_week_raw_scan_summary.csv", index=False)
if not filtered_chunks:
    raise ValueError("No rows were extracted for the validation window. Check encoded month selection and unix timestamp handling.")

filtered_raw = pd.concat(filtered_chunks, ignore_index=True)
filtered_raw = filtered_raw.groupby(["edp_site_id", "unix_time"], observed=True, as_index=False)["real_energy"].sum()
site_row_counts = filtered_raw.groupby("edp_site_id", observed=True).size().reset_index(name="raw_5min_rows")
site_row_counts.to_csv(AUDIT_DIR / "aedp_one_week_ac_load_net_site_row_counts.csv", index=False)
print(f"Filtered in-memory ac_load_net rows after grouping duplicates: {filtered_raw.shape[0]}")
print(f"Sites with ac_load_net rows: {site_row_counts.shape[0]}")
display(raw_scan_summary)


[done] edp_data_2021_0781620172.csv: chunks=13, rows=282194
Filtered in-memory ac_load_net rows after grouping duplicates: 280184
Sites with ac_load_net rows: 139


,filename,chunks,output_rows
0,edp_data_2021_0781620172.csv,13,282194


## 1.5 Build One-Week Per-Site 30-Minute Checkpoints

Each site with `ac_load_net` rows in the validation week is reconstructed over the native 5-minute grid, converted from energy to power, forward-filled if needed, and resampled to 30-minute mean power. Sites from the recovered 148-household cohort that do not have `ac_load_net` rows in this one-week window are audited separately instead of being treated as a fatal error.


In [5]:
def build_site_30min_checkpoint(site_id: str) -> Path:
    out_path = SITE_30MIN_DIR / f"{site_id}_30min.parquet"
    part = filtered_raw.loc[filtered_raw["edp_site_id"].eq(site_id)].copy()
    if part.empty:
        raise ValueError(f"No filtered ac_load_net rows found for {site_id}")

    part["datetime"] = pd.to_datetime(part["unix_time"], unit="s")
    native_energy = part.groupby("datetime", observed=True)["real_energy"].sum().reindex(INDEX_NATIVE)
    native_power = native_energy.div(1000.0).div(RAW_INTERVAL_HOURS).to_frame("netload_kW")
    native_power["netload_kW"] = native_power["netload_kW"].ffill()
    if native_power["netload_kW"].isna().any():
        raise ValueError(f"{site_id}: remaining missing values after forward-fill")

    site_30min = native_power["netload_kW"].resample("30min", label="left", closed="left").mean().reindex(INDEX_30MIN).reset_index()
    site_30min.insert(0, "edp_site_id", site_id)
    if site_30min["netload_kW"].isna().any():
        raise ValueError(f"{site_id}: 30-minute checkpoint contains missing values")
    site_30min.to_parquet(out_path, index=False)
    return out_path


available_site_ids = sorted(filtered_raw["edp_site_id"].unique())
checkpoint_site_ids = [site_id for site_id in site_ids if site_id in set(available_site_ids)]
missing_ac_load_net_sites = [site_id for site_id in site_ids if site_id not in set(available_site_ids)]

missing_site_audit = site_list.loc[site_list["edp_site_id"].isin(missing_ac_load_net_sites)].copy()
missing_site_audit["validation_note"] = "No ac_load_net rows extracted for this one-week validation window."
missing_site_audit.to_csv(AUDIT_DIR / "aedp_one_week_missing_ac_load_net_sites.csv", index=False)

checkpoint_rows = []
for site_id in checkpoint_site_ids:
    out_path = build_site_30min_checkpoint(site_id)
    checkpoint_rows.append({"edp_site_id": site_id, "site_30min_path": str(out_path), "rows": len(INDEX_30MIN)})
checkpoint_index = pd.DataFrame(checkpoint_rows)
checkpoint_index.to_csv(AUDIT_DIR / "aedp_one_week_site_checkpoint_index.csv", index=False)

print(f"Recovered cohort sites: {len(site_ids)}")
print(f"Sites with ac_load_net rows in validation week: {len(checkpoint_site_ids)}")
print(f"Recovered cohort sites without ac_load_net rows in validation week: {len(missing_ac_load_net_sites)}")
if missing_ac_load_net_sites:
    display(missing_site_audit[["edp_site_id", "date_of_first_data", "date_of_last_data", "validation_note"]])
print(f"Built one-week site checkpoints: {checkpoint_index.shape[0]}")
display(checkpoint_index.head())


Recovered cohort sites: 148
Sites with ac_load_net rows in validation week: 139
Recovered cohort sites without ac_load_net rows in validation week: 9


,edp_site_id,date_of_first_data,date_of_last_data,validation_note
6,S0358,2019-01-01,2024-08-02,No ac_load_net rows extracted for this one-wee...
21,S0156,2019-04-08,2024-08-30,No ac_load_net rows extracted for this one-wee...
31,S0162,2019-01-01,2024-08-30,No ac_load_net rows extracted for this one-wee...
33,S0195,2019-01-01,2024-08-30,No ac_load_net rows extracted for this one-wee...
49,S0137,2019-01-01,2024-09-30,No ac_load_net rows extracted for this one-wee...
117,S0075,2018-01-01,2024-09-30,No ac_load_net rows extracted for this one-wee...
134,W0231,2021-03-11,2024-09-30,No ac_load_net rows extracted for this one-wee...
136,S0167,2021-03-03,2024-09-30,No ac_load_net rows extracted for this one-wee...
147,S0081,2018-01-01,2024-09-30,No ac_load_net rows extracted for this one-wee...


Built one-week site checkpoints: 139


,edp_site_id,site_30min_path,rows
0,S0405,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,336
1,W0013,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,336
2,S0321,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,336
3,S0431,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,336
4,S0170,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,336


## 1.6 Validate Against Existing ds10 And ds11 For The Same Week

The one-week aggregate is rebuilt from the raw `ac_load_net` rows using the historical group-level order: sum the selected-site raw energy by timestamp first, convert to power, forward-fill the aggregate native grid, then resample to 30 minutes. This should reproduce the matching slice of `ds10` and `ds11` within rounding tolerance.

The notebook also reports how much this differs from summing the per-site 30-minute checkpoints. That diagnostic matters because per-site forward-fill before summing is not equivalent when some sites have gaps.


In [6]:
def load_site_series(site_id: str) -> pd.Series:
    path = SITE_30MIN_DIR / f"{site_id}_30min.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing site checkpoint: {path}")
    frame = pd.read_parquet(path)
    return frame.set_index("datetime")["netload_kW"].reindex(INDEX_30MIN)


def build_group_level_aggregate_from_filtered_raw() -> pd.Series:
    raw_parts = filtered_raw.copy()
    if raw_parts.empty:
        raise ValueError("filtered_raw is empty")
    raw_parts["datetime"] = pd.to_datetime(raw_parts["unix_time"], unit="s")
    native_energy = raw_parts.groupby("datetime", observed=True)["real_energy"].sum().reindex(INDEX_NATIVE)
    native_power = native_energy.div(1000.0).div(RAW_INTERVAL_HOURS).to_frame("netload_kW")
    native_power["netload_kW"] = native_power["netload_kW"].ffill()
    if native_power["netload_kW"].isna().any():
        raise ValueError("Group-level aggregate has missing values after forward-fill")
    return native_power["netload_kW"].resample("30min", label="left", closed="left").mean().reindex(INDEX_30MIN)


aggregate_30min = build_group_level_aggregate_from_filtered_raw()
repro_10 = aggregate_30min.to_frame("netload_kW")
repro_10.index.name = "datetime"

per_site_checkpoint_sum = sum((load_site_series(site_id) for site_id in checkpoint_site_ids), start=pd.Series(0.0, index=INDEX_30MIN))
per_site_vs_group_max_abs_diff = (per_site_checkpoint_sum - aggregate_30min).abs().max()
per_site_vs_group_mean_abs_diff = (per_site_checkpoint_sum - aggregate_30min).abs().mean()

existing_10 = pd.read_csv(DS10_PATH, parse_dates=["datetime"]).set_index("datetime").reindex(INDEX_30MIN)
existing_11 = pd.read_csv(DS11_PATH, parse_dates=["datetime"]).set_index("datetime").reindex(INDEX_30MIN)
if existing_10.isna().any().any():
    raise ValueError("Existing ds10 slice contains missing values after reindexing")
if existing_11.isna().any().any():
    raise ValueError("Existing ds11 slice contains missing values after reindexing")
weather = existing_11.drop(columns=["netload_kW"])
repro_11 = repro_10.merge(weather, left_index=True, right_index=True, how="left")

checks = {
    "ds10_30min_one_week": (repro_10.round(3), existing_10.round(3)),
    "ds11_30min_weather_one_week": (repro_11.round(3), existing_11.round(3)),
}
validation_rows = []
for name, (left, right) in checks.items():
    if list(left.columns) != list(right.columns):
        raise ValueError(f"{name}: column mismatch {list(left.columns)} != {list(right.columns)}")
    if not left.index.equals(right.index):
        raise ValueError(f"{name}: datetime index mismatch")
    max_abs_diff = (left - right).abs().max().max()
    mean_abs_diff = (left - right).abs().mean().mean()
    validation_rows.append({
        "dataset": name,
        "rows": left.shape[0],
        "recovered_cohort_sites": len(site_ids),
        "checkpoint_sites_with_ac_load_net": len(checkpoint_site_ids),
        "missing_ac_load_net_sites": len(missing_ac_load_net_sites),
        "group_level_vs_per_site_checkpoint_max_abs_diff": per_site_vs_group_max_abs_diff,
        "group_level_vs_per_site_checkpoint_mean_abs_diff": per_site_vs_group_mean_abs_diff,
        "max_abs_difference_after_rounding": max_abs_diff,
        "mean_abs_difference_after_rounding": mean_abs_diff,
    })
    if max_abs_diff > ROUNDING_TOLERANCE:
        raise AssertionError(f"{name}: max absolute difference {max_abs_diff} exceeds tolerance {ROUNDING_TOLERANCE}")

validation = pd.DataFrame(validation_rows)
validation.to_csv(AUDIT_DIR / "aedp_one_week_reproduction_validation.csv", index=False)
repro_10.reset_index().to_parquet(ONE_WEEK_WORKSPACE / "aedp_148hh_reproduced_aggregate_30min_one_week.parquet", index=False)
repro_11.reset_index().to_parquet(ONE_WEEK_WORKSPACE / "aedp_148hh_reproduced_aggregate_30min_with_weather_one_week.parquet", index=False)
display(validation)
print("AEDP one-week 30-minute aggregate reproduction passed validation using group-level raw aggregation.")


,dataset,rows,recovered_cohort_sites,checkpoint_sites_with_ac_load_net,missing_ac_load_net_sites,group_level_vs_per_site_checkpoint_max_abs_diff,group_level_vs_per_site_checkpoint_mean_abs_diff,max_abs_difference_after_rounding,mean_abs_difference_after_rounding
0,ds10_30min_one_week,336,148,139,9,3.317518,0.035806,0.0,0.0
1,ds11_30min_weather_one_week,336,148,139,9,3.317518,0.035806,0.0,0.0


AEDP one-week 30-minute aggregate reproduction passed validation using group-level raw aggregation.


## 1.7 Completion Notes

This notebook validates the reconstruction logic for one week only. It is intentionally not sufficient for final `ds25`-`ds36` generation, because notebook `2.1` builds full-period per-household checkpoints and notebook `2.2` creates the aggregation-level datasets.

For this validation week, the historical aggregate is reproduced by summing raw `ac_load_net` rows across the available cohort sites before forward-fill and 30-minute resampling. Any recovered sites without `ac_load_net` rows are written to the missing-site audit file so the cohort/accounting issue remains visible.

Important implementation note for future aggregation-level datasets: summing per-site 30-minute checkpoints can differ from group-level raw aggregation when individual sites have gaps, so final group datasets should preserve the historical group-level aggregation order wherever practical.
